# Practical Introduction to Hugging Face Pipelines
### Modern NLP Workflows in a Single Line of Code

The `transformers.pipeline` abstraction wraps the entire deep learning inference workflow into high-level calls:
1. **Preprocessing / Tokenization**: Converting raw text into model-ready tensor inputs.
2. **Forward Pass**: Running tensor operations through the pretrained transformer architecture.
3. **Post-processing**: Converting raw logits into human-interpretable labels, probabilities, or decoded text.

This notebook demonstrates **current essential modern NLP tasks** using up-to-date Hugging Face pipelines.

## 0. Setup and Environment Verification
First, make sure `transformers`, `torch`, and `accelerate` are installed. We also configure dynamic device selection (CUDA, Apple MPS, or CPU) and appropriate precision dtypes.

In [8]:
# HF access authorization

from hf_auth import init_hf

# Initialize the token securely before loading any models
init_hf()

✅ Hugging Face token successfully loaded!


True

In [9]:
import torch
from transformers import pipeline
from warnings import filterwarnings

filterwarnings("ignore")

# Automatic accelerator detection
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Active accelerator: {device}")
print(f"Target data type:   {dtype}")

Active accelerator: mps
Target data type:   torch.float32


In [10]:
# Check the the tasks supported by the pipeline API in the current version of transformers

from transformers.pipelines import SUPPORTED_TASKS

# Extract and sort the list of available tasks
available_tasks = sorted(list(SUPPORTED_TASKS.keys()))
for i in available_tasks:
    print(i,',')

any-to-any ,
audio-classification ,
automatic-speech-recognition ,
depth-estimation ,
document-question-answering ,
feature-extraction ,
fill-mask ,
image-classification ,
image-feature-extraction ,
image-segmentation ,
image-text-to-text ,
keypoint-matching ,
mask-generation ,
object-detection ,
table-question-answering ,
text-classification ,
text-generation ,
text-to-audio ,
token-classification ,
video-classification ,
zero-shot-audio-classification ,
zero-shot-classification ,
zero-shot-image-classification ,
zero-shot-object-detection ,


---
## Task 1: Text Classification (Sentiment Analysis)
**Goal**: Assign categorical labels with calibrated confidence probabilities to input sequences.

* **Default / Example Model**: `distilbert/distilbert-base-uncased-finetuned-sst-2-english`
* **Key Concept**: Pipelines accept Python lists of strings, automatically enabling batched tensor operations.

In [11]:
classifier = pipeline(
    task="text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

sample_reviews = [
    "The explanations in this NLP lecture made transformer architectures crystal clear!",
    "The code kept throwing obscure CUDA errors and lacked documentation."
]

predictions = classifier(sample_reviews)

print("--- Text Classification Results ---")
for text, pred in zip(sample_reviews, predictions):
    print(f"Text:  \"{text}\"")
    print(f"Label: {pred['label']} (Confidence: {pred['score']:.4f})\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- Text Classification Results ---
Text:  "The explanations in this NLP lecture made transformer architectures crystal clear!"
Label: POSITIVE (Confidence: 0.9993)

Text:  "The code kept throwing obscure CUDA errors and lacked documentation."
Label: NEGATIVE (Confidence: 0.9998)



---
## Task 2: Zero-Shot Classification
**Goal**: Categorize text into arbitrary, custom labels **without** fine-tuning or retraining the underlying network.

* **Model**: `facebook/bart-large-mnli`
* **Mechanism**: Reformulates classification as Natural Language Inference (NLI) by testing whether the text entails a hypothesis like *"This text is about {label}"*.

In [12]:
zero_shot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

sequence_to_classify = "The central bank decided to hold interest rates steady amid stubborn inflation readings."
candidate_labels = ["sports", "finance", "technology", "healthcare"]

zero_shot_res = zero_shot_classifier(
    sequence_to_classify,
    candidate_labels=candidate_labels,
    multi_label=False
)

print("--- Zero-Shot Classification Results ---")
print(f"Input: {zero_shot_res['sequence']}")
for label, score in zip(zero_shot_res['labels'], zero_shot_res['scores']):
    print(f"  • {label:12}: {score * 100:.2f}%")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

--- Zero-Shot Classification Results ---
Input: The central bank decided to hold interest rates steady amid stubborn inflation readings.
  • finance     : 82.04%
  • technology  : 8.35%
  • sports      : 5.38%
  • healthcare  : 4.23%


---
## Task 3: Token Classification (Named Entity Recognition - NER)
**Goal**: Classify individual token spans corresponding to real-world entities such as persons (`PER`), organizations (`ORG`), or locations (`LOC`).

* **Model**: `dslim/bert-base-NER`
* **Key Argument**: `aggregation_strategy="simple"` merges BPE/WordPiece subword tokens (e.g., `_Sat` + `ya`) into cohesive whole words.

In [13]:
ner_tagger = pipeline(
    task="token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device
)

ner_input = "Satya Nadella spoke about Microsoft's new research campus in Redmond, Washington."
entities = ner_tagger(ner_input)

print("--- Named Entity Recognition Results ---")
for ent in entities:
    print(f"Entity: {ent['word']:<14} | Type: {ent['entity_group']:<5} | Score: {ent['score']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Named Entity Recognition Results ---
Entity: Sa             | Type: PER   | Score: 0.9997
Entity: ##tya Nadella  | Type: PER   | Score: 0.9275
Entity: Microsoft      | Type: ORG   | Score: 0.9982
Entity: Redmond        | Type: LOC   | Score: 0.9671
Entity: Washington     | Type: LOC   | Score: 0.9983


---
## Task 4: Text Generation & Conversational Chat
**Goal**: Auto-regressively generate continuations and follow user dialogue instructions.

* **Model**: `Qwen/Qwen2.5-0.5B-Instruct` (compact, high-performing open-weights model)
* **Modern HF Standard**: Hugging Face deprecated the legacy `conversational` pipeline in favor of passing chat messages directly to `text-generation`. Chat templates format role-based dictionaries (`system`, `user`, `assistant`) automatically.

In [14]:
generator = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=dtype,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful and concise computer science tutor."},
    {"role": "user", "content": "Explain what tokenization is in 2 sentences."}
]

generation_output = generator(
    messages,
    max_new_tokens=60,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print("--- Text Generation Output ---")
# The pipeline appends the assistant reply to the end of the message list
assistant_reply = generation_output[0]["generated_text"][-1]["content"]
print(assistant_reply)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Text Generation Output ---
Tokenization is the process of breaking down text into individual words or tokens, which can then be analyzed for various purposes㌔. This involves separating punctuation marks from text to create meaningful units for further processing or analysis.


---
### Summary of Takeaways
1. **Uniform Interface**: Across every architecture (encoder-only, encoder-decoder, decoder-only), the call pattern remains `pipeline(task, model)(inputs)`.
2. **Subword Reassembly**: Token-level tasks benefit from `aggregation_strategy="simple"`.
3. **Chat Formatting**: Modern autoregressive models use `messages` lists of dictionaries rather than raw string concatenation.